In [ ]:
# @title Zelle 1: Setup, GitHub-Auth, agy CLI & OKF-Skill
import os, subprocess

# --- 1. Git & GitHub Auth ---
subprocess.run(["git", "config", "--global", "user.name", "Colab Automation Bot"], check=True)
subprocess.run(["git", "config", "--global", "user.email", "colab-automation-bot@users.noreply.github.com"], check=True)

from google.colab import userdata
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise RuntimeError("Secret GITHUB_TOKEN fehlt. Bitte in den Colab-Secrets anlegen.")
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

REPO = "rhigazi/newscheck"
WORKDIR = "/content/newscheck"
CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    print("Repository vorhanden -> aktualisiere (git pull) ...")
    subprocess.run(["git", "-C", WORKDIR, "pull", "origin", "main"], check=True)
else:
    print("Klone Repository ...")
    subprocess.run(["git", "clone", "--depth", "1", CLONE_URL, WORKDIR], check=True)

os.chdir(WORKDIR)

# --- 2. agy CLI Installation & Pfad-Erweiterung ---
print("Prüfe agy CLI Installation...")
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")
agy_path = subprocess.run(["which", "agy"], capture_output=True, text=True).stdout.strip()

if not agy_path:
    print("Installiere agy CLI...")
    subprocess.run("curl -fsSL https://antigravity.google/cli/install.sh | bash", shell=True, check=True)

print("ℹ️ Hinweis: Die agy-Authentifizierung wird vom Nutzer im Anschluss manuell durchgeführt.")

# --- 3. OKF Skill non-interaktiv installieren ---
print("Installiere OKF Skill (okf-knowledge-base)...")
env = os.environ.copy()
env["CI"] = "true"  # Zwingt npx/skills in den non-interaktiven Modus

skill_cmd = [
    "npx", "--yes", "skills", "add", 
    "https://github.com/inkeep/open-knowledge-skills", 
    "--skill", "okf-knowledge-base", 
    "--yes"
]
skill_result = subprocess.run(skill_cmd, env=env, capture_output=True, text=True)
print(skill_result.stdout)
if skill_result.returncode != 0:
    print("Hinweis bei Skill-Installation:", skill_result.stderr)

print("✅ Setup von Repository, agy und OKF-Skill erfolgreich abgeschlossen!")


In [ ]:
# @title Zelle 2: Marktdaten-Abruf (Alpha Vantage / DIA ETF)
import os, requests, pandas as pd

from google.colab import userdata
ALPHA_VANTAGE_API_KEY = userdata.get("ALPHA_VANTAGE_API_KEY")
if not ALPHA_VANTAGE_API_KEY:
    raise RuntimeError("Secret ALPHA_VANTAGE_API_KEY fehlt. Bitte in den Colab-Secrets anlegen.")
os.environ["ALPHA_VANTAGE_API_KEY"] = ALPHA_VANTAGE_API_KEY

SYMBOL = "DIA"  # SPDR Dow Jones Industrial Average ETF Trust (stabiler Ersatz für DJI-Index)
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": SYMBOL,
    "outputsize": "compact",
    "apikey": ALPHA_VANTAGE_API_KEY,
}

try:
    resp = requests.get("https://www.alphavantage.co/query", params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
except requests.RequestException as e:
    raise RuntimeError("Alpha Vantage Request fehlgeschlagen: " + str(e))

# --- Rate-Limit / API-Fehler sauber abfangen ---
if "Note" in data or "Information" in data:
    msg = data.get("Note") or data.get("Information")
    raise RuntimeError("Alpha Vantage Rate-Limit erreicht: " + str(msg))
if "Error Message" in data:
    raise RuntimeError("Alpha Vantage Fehler: " + str(data["Error Message"]))
if "Time Series (Daily)" not in data:
    raise RuntimeError("Unerwartete Antwort von Alpha Vantage: " + str(list(data.keys())))

ts = data["Time Series (Daily)"]
df = pd.DataFrame.from_dict(ts, orient="index")
df.index = pd.to_datetime(df.index)
df = df.sort_index()
df.columns = ["open", "high", "low", "close", "volume"]
df = df.astype({"open": float, "high": float, "low": float, "close": float, "volume": int})
df.index.name = "date"

print("📈 DIA-ETF-Tagesdaten erfolgreich geladen:", len(df), "Zeilen")
display(df.tail(10))


In [ ]:
# @title Zelle 3: News-Dateien einlesen
import os, glob

WORKDIR = "/content/newscheck"
news_files = sorted(glob.glob(os.path.join(WORKDIR, "*.txt")))
if not news_files:
    raise FileNotFoundError("Keine .txt-Tagesdateien in " + WORKDIR + " gefunden.")

news_archive = {}
for path in news_files:
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        news_archive[os.path.basename(path)] = f.read()

print(str(len(news_archive)) + " News-Dateien geladen:")
for _name, _text in news_archive.items():
    print("  - " + _name + ": " + str(len(_text)) + " Zeichen")


In [ ]:
# @title Zelle 4: Analyse & OKF-Regelsynthese via agy (OKF-Skill)
import os, subprocess, textwrap

# --- Token-Optimierung: News-Texte vor der LLM-Übergabe begrenzen ---
MAX_CHARS_PER_FILE = 4000
MAX_TOTAL_CHARS = 20000

def _truncate(text, limit):
    text = text.strip()
    return text[:limit] + ("\n...[gekürzt]" if len(text) > limit else "")

_snippets = []
_total = 0
for _name, _text in news_archive.items():
    if _total >= MAX_TOTAL_CHARS:
        break
    _snippet = _truncate(_text, MAX_CHARS_PER_FILE)
    _snippets.append("### " + _name + "\n" + _snippet)
    _total += len(_snippet)

news_block = "\n\n".join(_snippets)
market_block = df.tail(10).to_string()

prompt = textwrap.dedent(f"""
    Du bist ein Marktanalyst. Analysiere die menschlichen Verhaltensnarrative
    (z. B. Panik, Zinsangst, Erwartungshaltungen) in den Nachrichten und gleiche
    sie mit den DJIA-Kursdaten ab. Synthetisiere daraus OKF-Regeln (Open Knowledge
    Format v0.2) und lege/aktualisiere die OKF-Dateien im Verzeichnis ./okf-regeln/
    gemäß dem OKF-Skill.

    ## Marktdaten (DJIA, letzte 10 Handelstage)
    {market_block}

    ## Nachrichten-Snippets (token-optimiert)
    {news_block}
""").strip()

# --- agy (Google Antigravity CLI) mit installiertem OKF-Skill ausführen ---
cmd = ["agy", "run", "--skill", "okf-knowledge-base", "--prompt", prompt]
print("Starte agy-Analyse mit OKF-Skill ...")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("agy stderr:", result.stderr)
    raise RuntimeError("agy-Analyse fehlgeschlagen")


In [ ]:
# @title Zelle 5: Automatisierter Git Commit & Push
import os, subprocess, datetime

WORKDIR = "/content/newscheck"
os.chdir(WORKDIR)

def _git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True, check=True)

# --- Idempotenz: nur bei echten Änderungen committen ---
status = _git("status", "--porcelain").stdout.strip()
if not status:
    print("Keine Änderungen erkannt -> nichts zu committen (idempotent).")
else:
    print("Änderungen erkannt:")
    print(status)
    _git("add", "-A")
    _stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    _git("commit", "-m", "chore(okf): automatisiertes Regelupdate (" + _stamp + ")")
    push = _git("push", "origin", "main")
    print("Push erfolgreich:", push.stdout or "ok")
